# 📜 Pydantic y humanidades digitales

**Laboratorio de PLN — IFTS24**
Matías Barreto, 2026

**Encuentro 14 · Bloque opcional — 50 minutos**

---

## Objetivo

Usar Ollama + Pydantic para extraer datos estructurados de textos históricos no estructurados — el caso de uso de Digital Humanities.

## Al terminar este bloque vas a poder:

1. Declarar schemas de extracción con Pydantic y conectarlos a Ollama.
2. Procesar lotes de textos históricos y convertirlos en DataFrames limpios.
3. Visualizar redes de coocurrencia a partir de metadatos extraídos.

## ◈ Microglosario

| Término | Qué es en lenguaje llano |
|---|---|
| **Pydantic** | Librería de Python que valida datos contra un schema estricto — si el modelo devuelve algo incorrecto, falla en lugar de pasar silenciosamente. |
| **BaseModel** | Clase base de Pydantic de la que heredan tus schemas; define campos, tipos y restricciones. |
| **Granite 4** | Familia de modelos IBM open source, eficientes para extracción de texto estructurado. |
| **UMAP** | Algoritmo de reducción de dimensionalidad que proyecta embeddings de 768D a 2D para visualización. |
| **Red de coocurrencia** | Grafo donde los nodos son entidades y las aristas representan apariciones conjuntas en documentos. |

> **◈ Importante — este notebook corre en local, no en Google Colab.**
>
> Necesita acceso a `localhost:11434` donde corre Ollama.
> Antes de empezar:
> 1. Asegurate de que Ollama esté corriendo: `ollama serve`
> 2. Verificá que el modelo esté disponible: `ollama list`
> 3. Si no ves `granite4:latest`, descargalo: `ollama pull granite4:latest`

In [ ]:
# Instalación
!uv pip install ollama

In [ ]:
# Importación
import ollama as ol

## El problema de Digital Humanities

### Analogía

Imaginate que tenés miles de registros bibliográficos del siglo XVII escritos así:

> *'impreso por los cesionarios de Rich. y Edw. Atkins, Esq.; y ha de venderse por Tho. Burrel en la Golden-Ball...'*

Una expresión regular podría capturar algo, pero hay demasiada variación. El LLM puede leer eso y extraer: impresor = Atkins, librero = Tho. Burrel, lugar = Fleetstreet.

### Dónde vive esto en el mundo real

Proyectos como el English Short-Title Catalog (ESTC) y la Poetry Foundation tienen millones de registros con metadatos embebidos en texto libre. Investigadores de historia del libro, sociología de la lectura y redes editoriales usan este patrón — Ollama local + Pydantic — para procesar corpus sin enviar datos sensibles a APIs externas y sin costo de inferencia.

**Importante:** este notebook corre en local. No uses Google Colab — necesita conexión a `localhost:11434` donde corre Ollama.

### ✎ Para pensar

- ¿Por qué no alcanza con pedirle al modelo que devuelva JSON en el prompt? ¿Qué garantía extra da Pydantic?
- Si el campo `fecha` es `Optional[int]`, ¿qué pasa cuando el texto no tiene fecha? ¿Y si el modelo inventa una?

## Parte 1 — Chat general y embeddings

Primero verificamos que Ollama corre y el modelo responde.

```bash
# Terminal
ollama pull granite4:latest
ollama serve
```

In [ ]:
response = ol.chat( # Guardamos la salida de tu chat en una variable de respuesta
    model='ibm/granite4:1b', # Declaramos el modelo que vas a utilizar
    messages = [{'role':'user','content':'Escribi un soneto sobre un tema de tu eleccion.'}],
    options = {'temperature': 0} # Controla cuan determinista es la respuesta del modelo; 0 es lo mas determinista
    # Nota: a veces una temperatura mas alta es mejor para ciertas tareas. Considera probar diferentes valores.
)

print(response.message.content)


### Visualizando similitud semántica con UMAP

Tomamos 12 poemas de tres temas (muerte, amor, naturaleza) y los proyectamos a 2D para ver si el modelo agrupa los textos por significado.

In [ ]:
# Usamos la funcion embed en lugar de chat
embed_response = ol.embed(model='ibm/granite4:1b', input='Aca hay un texto para vectorizar.')
embeddings = embed_response.embeddings
print(embeddings)


In [ ]:
texts = [
    # Muerte
    "Porque no pude detenerme ante la Muerte -",                  # Emily Dickinson
    "No entres docilmente en esa buena noche,",                   # Dylan Thomas
    "Muerte, no te envanezcas, aunque algunos te hayan llamado,", # John Donne
    "La tumba es un lugar bello y privado,",                      # Andrew Marvell

    # Amor
    "Como te amo? Dejame contar las maneras.",                    # Elizabeth Barrett Browning
    "Yo te ame primero: pero despues tu amor,",                   # Christina Rossetti
    "El amor no es amor si cambia cuando halla mudanza,",         # Shakespeare (Soneto 116)
    "Mi amor es como una rosa roja, roja,",                       # Robert Burns

    # Naturaleza
    "Vague solitario como una nube",                              # William Wordsworth
    "Los bosques son hermosos, oscuros y hondos,",                # Robert Frost
    "Una multitud de dorados narcisos;",                          # William Wordsworth
    "Sube la marea, baja la marea,",                              # Henry Wadsworth Longfellow
]


# Vectorizamos los textos (incrustaciones)
embed_response = ol.embed(model='ibm/granite4:1b', input=texts)
embeddings = embed_response.embeddings


In [ ]:
# Instalacion
!uv pip install umap-learn


In [ ]:
import umap.umap_ as umap

In [ ]:
reducer = umap.UMAP(n_neighbors=4)
umap_embeddings = reducer.fit_transform(embeddings)


In [ ]:
import pandas as pd

# Formateamos las incrustaciones y los textos en un DataFrame de pandas para graficar
df_to_plot = pd.DataFrame({"x": umap_embeddings[:, 0],
                           "y": umap_embeddings[:, 1],
                           "texto": texts,
                          })

df_to_plot.head()


In [ ]:
# Graficamos los documentos; creamos etiquetas emergentes con informacion al pasar el mouse
import altair as alt

# Creamos el grafico
alt.Chart(df_to_plot,
          # Titulo del grafico / tamano de los puntos
          title="Similitud textual").mark_circle(size=200).encode(
    # Definimos la categoria del eje x, que es la columna "x" del DataFrame
    alt.X('x',
        scale=alt.Scale(zero=False),
        # Como las posiciones de los puntos y ejes no importan, removemos las etiquetas de los ejes
        axis=alt.Axis(labels=False),
    ),
    # Definimos la categoria del eje y, que es la columna "y" del DataFrame
    alt.Y("y",
        # Como las posiciones de los puntos y ejes no importan, removemos las etiquetas de los ejes
          axis=alt.Axis(labels=False)),
    # Definimos las categorias que aparecen en la etiqueta flotante
    tooltip=['texto'],
    ).interactive().properties(
   # width=500,
   # height=500
)


### ✎ Para pensar

Mirá el mapa generado por UMAP:

- ¿Los poemas del mismo tema (muerte, amor, naturaleza) quedaron agrupados cerca entre sí?
- Si dos poemas de temas distintos quedaron muy cerca, ¿qué podría explicar esa proximidad?
- ¿Qué nos dice la distribución espacial sobre cómo el modelo entiende el significado del texto?

### ✎ Para pensar

- ¿Los poemas del mismo tema quedaron cerca en el mapa UMAP? ¿Qué nos dice eso sobre los embeddings?
- Si dos poemas de temas distintos quedaran muy cerca, ¿qué podría significar?

## Parte 2 — Extracción con Pydantic: Poetry Foundation

### El patrón

1. Definís una clase Pydantic con los campos que querés extraer
2. La pasás a Ollama como `format=MiClase.model_json_schema()`
3. Ollama fuerza al modelo a generar solo JSON que cumpla ese schema

El resultado: datos listos para un DataFrame, sin parsing manual.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class FormatoRespuestaPoema(BaseModel):
    """Usa siempre esta herramienta para estructurar tu respuesta para la persona usuaria."""

    nombre_derechos: Optional[str] = Field(description="Nombre de la persona o entidad titular de los derechos de autor.")

    titulo_poema: Optional[str] = Field(description="Titulo del poema de origen.")

    publicacion: Optional[str] = Field(description="Libro, revista u otra publicacion en la que aparecio el poema de origen.")

    editorial: Optional[str] = Field(description="Editorial del libro, revista o publicacion.")

    fecha: Optional[int] = Field(description="Ano en que se emitieron los derechos de autor o en que se publico la fuente.")


In [ ]:
poetry_df = pd.read_csv(
    "https://raw.githubusercontent.com/melaniewalsh/whitespace-poetry/refs/heads/main/public_domain_poems-whitespace.csv",
    usecols=["poem_name", "poet_name", "poem_copyright_and_source"]
).rename(
    columns={
        "poem_name": "titulo_poema",
        "poet_name": "nombre_poeta",
        "poem_copyright_and_source": "derechos_y_fuente"
    }
)


In [ ]:
poetry_df = poetry_df[[ 'titulo_poema', 'nombre_poeta', 'derechos_y_fuente']]


In [ ]:
poetry_df

In [ ]:
test_poetry = poetry_df["derechos_y_fuente"].to_list()


In [ ]:
import json

### Paso previo — probando con un solo registro

Antes de procesar los 30 registros en lote, ejecutamos el extractor con un único ejemplo para verificar que el modelo devuelve el schema correcto.

In [ ]:
# Probamos con el primer registro antes de correr el lote completo
ejemplo = test_poetry[0]
print("Texto original:")
print(ejemplo)
print()

respuesta_prueba = ol.chat(
    model='gemma4:e2b',
    messages=[{'role': 'user',
               'content': f"Extrae la información de esta declaración de derechos en JSON: {ejemplo}\n\nDevolvé SOLO el JSON."}],
    format=FormatoRespuestaPoema.model_json_schema(),
    options={'temperature': 0.0},
)

print("Respuesta estructurada:")
print(respuesta_prueba.message.content)
print()
print("✓ Si el JSON tiene sentido, el bucle completo va a funcionar.")
print("  Si hay errores, mejor detectarlos acá antes de procesar 30 registros.")

In [ ]:
parsed_copyright = [] # Una lista vacia para guardar resultados

for copyright_statement in test_poetry[:10]: # Procesamos solo los primeros ejemplos, por ahora

    response = ol.chat(
      model='gemma4:e2b',
      messages=[{'role': 'user',
                 'content':
                 f"""
                  Vas a recibir una declaracion de derechos de autor.

                  A veces tendra informacion detallada completa (nombre_derechos, titulo_poema, publicacion, editorial, fecha).
                  A veces solo tendra el nombre de la publicacion y la fecha de la publicacion.

                  Devolve un diccionario JSON con SOLO los campos que puedas encontrar.

                  Si falta un campo, simplemente omitilo.

                  Aca tenes dos EJEMPLOS:

                  ---

                  Entrada:
                  Poesia (diciembre de 2024)

                  Salida:
                  {{
                    "publicacion": "Poesia",
                    "fecha": 2024
                  }}

                  ---
                  Entrada:
                  Credito de derechos de autor: William Butler Yeats, "La noche de todas las almas" de "Siete poemas y un fragmento". Dundrum: The Cuala Press, 1922. Dominio publico.', 'Fuente: Siete poemas y un fragmento (The Cuala Press, 1922)

                  Salida:
                  {{
                    "nombre_derechos": "William Butler Yeats",
                    "titulo_poema": "La noche de todas las almas",
                    "publicacion": "Siete poemas y un fragmento",
                    "editorial": "The Cuala Press",
                    "fecha": 1922
                  }}
                  ---

                  Ahora, esta es la entrada real:

                  {copyright_statement}

                  Salida:"""}],
      format=FormatoRespuestaPoema.model_json_schema(),  # Usamos Pydantic para generar el esquema o format=schema
      options={'temperature': 0.0},  # Hacemos que las respuestas sean mas deterministas
    )

    # Imprimimos la declaracion de derechos de autor y la respuesta
    print(" Original: ", copyright_statement, "\n", "Estructurado: ", response.message.content, "\n✦")
    # Agregamos cada resultado a la lista mayor
    parsed_copyright.append(json.loads(response.message.content)) # Usamos json para convertir los resultados a un diccionario


In [ ]:
# Ensanchamos las columnas para ver toda la informacion
pd.options.display.max_colwidth = 300

# Creamos el DataFrame procesado
parsed_df = pd.DataFrame(parsed_copyright)

# Tomamos las primeras 30 filas del DataFrame original
poetry_df_small = poetry_df.iloc[:30].copy()

# Unimos los dos
poetry_df_merged = pd.concat([poetry_df_small.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)

# Vemos el resultado
poetry_df_merged


In [ ]:
import matplotlib.pyplot as plt

# Creamos un histograma de las fechas
parsed_df['fecha'].dropna().hist(
    bins=60,
    # Rango del eje principal, del 1900 al 2025
    range=(1900, 2025),
    figsize=(10, 5),
    color='skyblue'
)

plt.show()


### ✎ Para pensar

- ¿Por qué marcamos `fecha` como `Optional[int]` en lugar de `str`?
- El notebook menciona que el modelo puede confundir roles ('Alfred A. Knopf' como editorial vs sello). ¿Cómo lo detectarías y corregirías?

## Parte 3 — Pies de imprenta del siglo XVII

Ahora aplicamos el mismo patrón a un corpus histórico más difícil: registros bibliográficos del ESTC con variaciones ortográficas y roles implícitos.

Aquí usamos un **system prompt** detallado en lugar de ejemplos en el `user` — las reglas de clasificación (impresor / editor / librero / lugar) son más complejas.

In [ ]:
# Leemos los datos como CSV
# Este es un conjunto con ejemplos que a nuestros metodos les costo procesar
catalog = pd.read_csv(
    "https://raw.githubusercontent.com/melaniewalsh/AI-4-Humanists/refs/heads/main/data/sample_imprints.csv",
    usecols=["estcNO_estccatalogue", "imprint"]
).rename(columns={"imprint": "pie_de_imprenta"})
catalog = catalog.set_index('estcNO_estccatalogue')

# Obtenemos un subconjunto mas pequeno para ensayo
test = catalog[:30]
test_imprints = test['pie_de_imprenta'].to_dict()
test_imprints = [f"ESTC {k}: {v}" for k, v in test_imprints.items()]


In [ ]:
from typing import Literal

class InformacionEntidad(BaseModel):
    nombre: str
    # La clase Literal permite especificar que un campo solo contenga ciertos valores
    tipo_entidad: Literal['impresor', 'editor', 'librero', 'lugar']
    id_estc: str = Field(description="Numero de identificacion del texto original")

class ListaEntidades(BaseModel):
    """Usa siempre esta herramienta para estructurar tu respuesta para la persona usuaria."""
    entidades: list[InformacionEntidad]


In [ ]:
response = ol.chat(
  model='gemma4:e2b',
  messages=[{'role': 'system', 'content': f"""
             Sos especialista en bibliografia del siglo XVII y queres identificar quien imprimio, publico, vendio y financio un gran conjunto de libros, asi como donde se imprimieron o vendieron esos libros.

             Extrae todas las entidades que encuentres en los pies de imprenta y usa las siguientes reglas para asignar tipos y un numero de identificacion:

             impresor: suele aparecer despues de "by" o "printed by"
             editor: suele aparecer despues de "for" o "printed for"
             librero: suele aparecer despues de "to be sold by" o cerca del nombre de un lugar o de la palabra "sold"
             lugar: cualquier nombre de lugar, ya sea donde el libro fue impreso o donde sera vendido
             id_estc: el numero de identificacion del pie de imprenta original. Nunca cambies ese numero y mantenelo asociado al nombre correcto extraido.

             Presta mucha atencion al tipo que le corresponde a cada entidad. Los nombres suelen tener dos palabras o iniciales. No te detengas en el primer nombre que encuentres.

             Si una persona tiene multiples roles (por ejemplo, impresor y librero), listala dos veces.
             """},{'role': 'user', 'content':f"{test_imprints}"
            }],
  format=ListaEntidades.model_json_schema(),  # Usamos Pydantic para generar el esquema, o definimos format='schema'
  options={'temperature': 0},  # Hacemos que las respuestas sean mas deterministas
)


In [ ]:
pd.set_option('display.max_colwidth', None)
parsed_imprint = pd.DataFrame(json.loads(response.message.content)['entidades']).set_index('id_estc')
parsed_imprint = parsed_imprint.join(test[['pie_de_imprenta']], how="right")
parsed_imprint


## Parte 4 — Redes de distribución editorial

Con los datos estructurados podemos construir un grafo de coocurrencia: dos personas están conectadas si aparecen juntas en el mismo pie de imprenta. El peso de la arista es la cantidad de coproducciones.

In [ ]:
!pip install --upgrade networkx

In [ ]:
import networkx as nx
from itertools import combinations

# Filtramos solo a las personas (excluyendo lugares)
people_df = parsed_imprint[parsed_imprint['tipo_entidad'].isin(['impresor', 'editor', 'librero'])].copy()

# Agrupamos por id_estc
grouped = people_df.groupby(people_df.index)

# Inicializamos la lista de conexiones (aristas)
edges = []

# Para cada ID de ESTC, obtenemos los pares unicos de personas
for estc_id, group in grouped:
    nombres = group['nombre'].unique()
    if len(nombres) > 1:
        edges.extend(combinations(sorted(nombres), 2))  # Ordenamos para que las combinaciones sean consistentes

# Contamos las apariciones de cada conexion (cuantos IDs comparten)
from collections import Counter
edge_counts = Counter(edges)

# Convertimos a DataFrame
weighted_edges_df = pd.DataFrame(
    [(a, b, w) for (a, b), w in edge_counts.items()],
    columns=['origen', 'destino', 'peso']
)


In [ ]:
weighted_edges_df

In [ ]:
# Creamos la red base con los origenes, destinos y pesos de enlace
G = nx.from_pandas_edgelist(weighted_edges_df, 'origen', 'destino', 'peso')

# Graficamos la red con etiquetas y colores
pos = nx.spring_layout(G, k=1)

plt.figure(figsize=(4,4))

nx.draw(G, pos=pos, with_labels=True, node_color='skyblue', width=1, font_size=6)


### 🐛 Laboratorio de Rotura

El schema de abajo tiene todos los campos **requeridos** (sin `Optional`). Antes de ejecutarlo, predecí: ¿qué va a pasar cuando el texto de entrada no tenga, por ejemplo, fecha de publicación?

```python
class SchemaEstricto(BaseModel):
    titulo: str       # requerido
    fecha: int        # requerido
    editorial: str    # requerido
```

Ejecutá la celda siguiente con un texto que deliberadamente no tiene todos los campos.

- ¿Qué esperabas?
- ¿Qué pasó en realidad?
- ¿Por qué `Optional` es la elección correcta para textos históricos variables?

No mires la explicación todavía.

In [ ]:
from pydantic import BaseModel, ValidationError

# ── Momento 1 — Schema sin Optional ──
class SchemaEstricto(BaseModel):
    titulo: str
    fecha: int        # ❌ requerido — qué pasa si el texto no tiene fecha
    editorial: str    # ❌ requerido

texto_sin_fecha = "Poesia (publicado por Editorial Sur)"

respuesta_rota = ol.chat(
    model='granite4:latest',
    messages=[{'role': 'user',
               'content': f"Extrae titulo, fecha y editorial de: {texto_sin_fecha}. Solo JSON."}],
    format=SchemaEstricto.model_json_schema(),
    options={'temperature': 0.0},
)

print("Respuesta del modelo (puede inventar la fecha):")
print(respuesta_rota.message.content)
print()

# ── Momento 2 — Con Optional, el modelo omite lo que no encuentra ──
from typing import Optional as Opt

class SchemaFlexible(BaseModel):
    titulo: Opt[str] = None
    fecha: Opt[int] = None      # ✓ Optional — si no está, queda None
    editorial: Opt[str] = None  # ✓ Optional

respuesta_flexible = ol.chat(
    model='granite4:latest',
    messages=[{'role': 'user',
               'content': f"Extrae titulo, fecha y editorial de: {texto_sin_fecha}. Solo JSON."}],
    format=SchemaFlexible.model_json_schema(),
    options={'temperature': 0.0},
)

print("Con Optional (la fecha debería ser null, no inventada):")
print(respuesta_flexible.message.content)
print()
print("◈ Con campos requeridos, el modelo puede *inventar* datos que no están.")
print("  Con Optional, los deja en null. Para corpus históricos incompletos, Optional es la regla.")

> **✎ Mentor reversible (bonus).** Hasta acá fuiste quien aprende. Ahora dalo vuelta: mirá la clase `FormatoRespuestaPoema` que definimos y reescribí su docstring —la que dice "Usa siempre esta herramienta..."— con palabras que le expliquen a alguien que nunca vio Pydantic qué hace cada campo y por qué algunos son `Optional`. Si esa persona imaginaria lo entiende, es porque vos ya lo entendiste de verdad.

In [ ]:
# --- Espacio de practica ---
#
# Adapta el extractor de Poetry Foundation a un dataset propio:
#   1. Buscá un dataset con texto no estructurado (puede ser de Wikipedia, CONICET, etc.)
#   2. Definí los campos que querés extraer como clase Pydantic
#   3. Escribí el system prompt con reglas de clasificación
#   4. Procesá 10-20 registros y revisá manualmente la precisión
#
# Bonus: construi un grafo de coocurrencia con los datos extraidos
#
# Pega aca tu codigo
pass

## Cierre del bloque

| Concepto | Qué aprendiste |
|---|---|
| **Pydantic BaseModel** | Define el schema y valida que el JSON cumpla los tipos |
| **format= en ol.chat()** | Fuerza al modelo a respetar el schema de Pydantic |
| **Optional** | Permite que un campo quede vacío sin romper la extracción |
| **System prompt para NER** | Reglas de clasificación en el rol system, ejemplos en user |
| **NetworkX** | Convierte datos tabulares en grafos de relaciones visualizables |

Este patrón — Ollama local + Pydantic + pipeline de procesamiento — es exactamente lo que usan equipos de investigación para procesar corpus históricos o legales sin enviar datos sensibles a APIs externas.

### 🧭 Diario de Navegación

Cerramos mirando hacia adentro, no hacia el código. Respondé en una o dos líneas, para vos:

1. ¿Qué concepto de hoy te costó más encajar en la cabeza? ¿Por qué creés que se resistió?
2. Si tuvieras que explicarle este cuaderno a un colega en lo que dura un viaje en ascensor, ¿qué le dirías?

No hay respuesta correcta. Lo que escribás es el mapa de tu propio recorrido.